# 01. BTC / ETH OHLCV 데이터 수집

**출처**: yfinance (Yahoo Finance)  
**수집 대상**: BTC-USD, ETH-USD  
**기간**: 2019-11-22 ~ 현재  
**저장 위치**: `data/raw/BTC-USD_historical_data.csv`, `ETH-USD_historical_data.csv`

**수집 이유**:  
- BTC/ETH는 스테이블코인 디페깅과 밀접한 관계 → 시장 전체 충격 지표  
- BTC 급락 시 USDT/USDC 거래량 급등, DAI는 ETH 담보 비율 위험 증가

In [ ]:
import yfinance as yf
import pandas as pd
import os

# 저장 경로 설정 (notebooks/ 기준 상대경로)
SAVE_DIR   = os.path.join('..', 'data', 'raw')
START_DATE = '2019-11-22'  # 프로젝트 분석 시작일

# 수집 대상 코인: {심볼: Yahoo Finance 티커}
COINS = {
    'BTC': 'BTC-USD',
    'ETH': 'ETH-USD'
}

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)

for symbol, ticker in COINS.items():
    print(f'수집 중: {symbol} ({ticker})...')

    # yfinance로 일별 OHLCV 다운로드
    df = yf.download(ticker, start=START_DATE, progress=False)

    # yfinance 멀티인덱스 컬럼 → 단일 컬럼으로 변환
    # yfinance 최신 버전에서 (Open, BTC-USD) 형태로 반환되는 문제 처리
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    # 컬럼명에 코인 심볼 추가 (BTC, ETH 구분용)
    df = df.rename(columns={
        'Open':   f'Open.{symbol.lower()}',
        'High':   f'High.{symbol.lower()}',
        'Low':    f'Low.{symbol.lower()}',
        'Close':  f'Close.{symbol.lower()}',
        'Volume': f'Volume.{symbol.lower()}'
    })

    # 인덱스(Date)를 컬럼으로 변환, 날짜를 YYYY-MM-DD 형식으로 통일
    df.index.name = 'Date'
    df = df.reset_index()
    df['Date'] = df['Date'].astype(str).str[:10]

    # CSV 저장
    save_path = os.path.join(SAVE_DIR, f'{symbol}-USD_historical_data.csv')
    df.to_csv(save_path, index=False)
    print(f'  완료: {len(df)}개 행 ({df["Date"].min()} ~ {df["Date"].max()})')
    print(f'  저장: {save_path}')

In [ ]:
# 수집 결과 확인
btc = pd.read_csv(os.path.join(SAVE_DIR, 'BTC-USD_historical_data.csv'))
eth = pd.read_csv(os.path.join(SAVE_DIR, 'ETH-USD_historical_data.csv'))

print('BTC:', btc.shape)
print(btc.head(3))
print()
print('ETH:', eth.shape)
print(eth.head(3))